# Qwen3.5 2B LoRA — v5 resumable training

This notebook trains the reviewed v5 generic requirements extractor on a T4 or better GPU. It uses a new Drive folder, pins all source revisions, verifies the frozen corpus hash, saves a complete checkpoint to Drive every 25 optimizer steps, and resumes only from a compatible complete checkpoint.

**Before running:** choose **Runtime → Change runtime type → T4 GPU**. Run cells in order. The sealed final test is never loaded. The user confirmed human review on 2026-09-27. The earlier interrupted run reached step 95 but produced no checkpoint, so this replacement run starts at step 0 in a new folder. The earlier pilot behavior comparison remains incomplete, so this notebook records a training-only protocol override; it does not declare a paper-quality model winner.


In [ ]:
from pathlib import Path
from google.colab import drive
import json, subprocess, sys

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

RUN = Path('/content/drive/MyDrive/FYP-model-runs/qwen35-2b-lora-v5-full-20260927-r2')
RUN.mkdir(parents=True, exist_ok=True)

PROJECT = Path('/content/slm-v5-full')
FPY = Path('/content/fpy-v5-full')
REPO_COMMIT = 'c1466082ab29544540d4687b1120b96a6a36f9a7'
FPY_COMMIT = '04d52c015d1e3ecdefe92b87116f209361509b4b'
MODEL = 'Qwen/Qwen3.5-2B'
MODEL_REVISION = '15852e8c16360a2fea060d615a32b45270f8a8fc'
EXPECTED_MANIFEST_SHA256 = '7e3c6f83bd39a73a386dbbf5e2a7932c45b68af46eaf6641e56474f2cc9dd021'
MAX_LENGTH = 3584

try:
    gpu = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        text=True,
    ).strip()
except Exception as exc:
    raise RuntimeError('A CUDA GPU runtime is required. Select T4 GPU and reconnect.') from exc
print('GPU:', gpu)
print('Drive run:', RUN)


In [ ]:
import json, subprocess, sys
from pathlib import Path

def command(*args):
    print('Running:', *args)
    subprocess.run([str(x) for x in args], check=True)

if not (PROJECT / '.git').is_dir():
    command('git', 'clone', 'https://github.com/AbdullahUsman0/SLM-FineTuning-Testing.git', PROJECT)
command('git', '-C', PROJECT, 'fetch', 'origin', 'experiment/v5-grounded-20260919')
command('git', '-C', PROJECT, 'checkout', '--detach', REPO_COMMIT)
actual_repo = subprocess.check_output(
    ['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_repo == REPO_COMMIT

if not (FPY / '.git').is_dir():
    command('git', 'clone', 'https://github.com/int-abd-5/fpy.git', FPY)
command('git', '-C', FPY, 'fetch', 'origin', FPY_COMMIT)
command('git', '-C', FPY, 'checkout', '--detach', FPY_COMMIT)
actual_fpy = subprocess.check_output(
    ['git', '-C', str(FPY), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_fpy == FPY_COMMIT

command(sys.executable, '-m', 'pip', 'install', '-r', PROJECT / 'training/requirements.txt')
command(sys.executable, '-m', 'pip', 'install', '-e', FPY)

source_lock = {
    'repository_commit': REPO_COMMIT,
    'fpy_commit': FPY_COMMIT,
    'model': MODEL,
    'model_revision': MODEL_REVISION,
    'license': 'Apache-2.0',
}
lock_path = RUN / 'source-lock.json'
if lock_path.exists():
    assert json.loads(lock_path.read_text()) == source_lock, 'Existing run has a different source lock'
else:
    lock_path.write_text(json.dumps(source_lock, indent=2) + '\n')

with (RUN / 'pip-freeze.txt').open('w') as handle:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=handle, check=True)
print(json.dumps(source_lock, indent=2))


In [ ]:
import hashlib, json, shutil, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path

CORPUS = PROJECT / 'corpus-v5/v5-20260919-r1'
subprocess.run([
    sys.executable,
    str(PROJECT / 'scripts/verify-corpus-v5.py'),
    '--output',
    str(CORPUS),
], cwd=PROJECT, check=True)

manifest_hash = hashlib.sha256((CORPUS / 'manifest.json').read_bytes()).hexdigest()
assert manifest_hash == EXPECTED_MANIFEST_SHA256, f'Unexpected corpus manifest: {manifest_hash}'

inputs = RUN / 'inputs'
inputs.mkdir(exist_ok=True)
input_hashes = {}
for name in ('train', 'validation'):
    source = CORPUS / 'sft' / f'{name}.jsonl'
    destination = inputs / f'{name}.jsonl'
    if destination.exists():
        assert destination.read_bytes() == source.read_bytes(), f'{name} input changed'
    else:
        shutil.copy2(source, destination)
    input_hashes[name] = hashlib.sha256(destination.read_bytes()).hexdigest()

preflight = {
    'corpus_manifest_sha256': manifest_hash,
    'verified_max_length': MAX_LENGTH,
    'length_measurement_source': (
        'v5 gate preflight: train max 3529, validation max 3460; rounded to 3584'
    ),
    'input_sha256': input_hashes,
}
preflight_path = RUN / 'length-preflight.json'
if preflight_path.exists():
    assert json.loads(preflight_path.read_text()) == preflight
else:
    preflight_path.write_text(json.dumps(preflight, indent=2) + '\n')

# This records the user's explicit confirmation. It does not invent an independent review ledger.
approval = {
    'corpus_manifest_sha256': manifest_hash,
    'reviewer': 'Abdullah Usman',
    'review_confirmed_at_utc': datetime.now(timezone.utc).isoformat(),
    'human_review_confirmed_by_user': True,
    'train_sample_approved': True,
    'validation_approved': True,
    'compute_run_authorized_by_user': True,
    'pilot_behavior_approved': False,
    'final_test_authorized': False,
    'protocol_deviation': (
        'User requested immediate full training before completion of the matched pilot behavior comparison.'
    ),
    'claim_limit': (
        'Training authorization only; model selection and paper-quality claims still require matched validation evaluation.'
    ),
}
approval_path = RUN / 'human-review-approval.json'
if approval_path.exists():
    previous = json.loads(approval_path.read_text())
    assert previous['corpus_manifest_sha256'] == manifest_hash
    assert previous['human_review_confirmed_by_user'] is True
else:
    approval_path.write_text(json.dumps(approval, indent=2) + '\n')

print(json.dumps(preflight, indent=2))
print('Approval recorded:', approval_path)
print('Final sealed test remains untouched.')


In [ ]:
import subprocess, sys, time
from pathlib import Path

output = RUN / 'full'
output.mkdir(exist_ok=True)
log_path = RUN / 'full-console.log'

args = [
    sys.executable,
    'training/train_lora.py',
    '--model', MODEL,
    '--revision', MODEL_REVISION,
    '--train', str(RUN / 'inputs/train.jsonl'),
    '--validation', str(RUN / 'inputs/validation.jsonl'),
    '--output', str(output),
    '--epochs', '2',
    '--save-steps', '25',
    '--learning-rate', '5e-5',
    '--early-stopping-patience', '0',
    '--max-length', str(MAX_LENGTH),
    '--resume-from-checkpoint', 'auto',
]

print('Training command:', ' '.join(map(str, args)))
print('Log:', log_path)
start = time.monotonic()
with log_path.open('a', encoding='utf-8', buffering=1) as log:
    log.write('\n=== FULL TRAINING START OR RESUME ===\n')
    process = subprocess.Popen(
        args,
        cwd=PROJECT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log.write(line)
    return_code = process.wait()
    elapsed = time.monotonic() - start
    log.write(f'\nreturn_code={return_code} elapsed_seconds={elapsed}\n')

print('Elapsed seconds:', elapsed)
print('Exit code:', return_code)
if return_code != 0:
    raise RuntimeError(
        'Training stopped. Preserve the Drive folder, reconnect a GPU, and rerun this same cell to resume.'
    )
print('Training completed. All 25-step durability checkpoints remain in:', output)


In [ ]:
import hashlib, json
from datetime import datetime, timezone
from pathlib import Path

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

output = RUN / 'full'
best = output / 'best-adapter'
assert best.is_dir(), 'best-adapter was not produced'
required = [best / 'adapter_model.safetensors', best / 'adapter_config.json']
for path in required:
    assert path.is_file(), f'Missing {path}'

checkpoints = sorted(
    p.name
    for p in output.glob('checkpoint-*')
    if p.is_dir() and (p / 'checkpoint-complete.json').is_file()
)
summary = {
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'run': str(RUN),
    'repository_commit': REPO_COMMIT,
    'model': MODEL,
    'model_revision': MODEL_REVISION,
    'corpus_manifest_sha256': EXPECTED_MANIFEST_SHA256,
    'epochs_requested': 2,
    'checkpoint_interval_steps': 25,
    'learning_rate': 5e-5,
    'max_length': MAX_LENGTH,
    'complete_checkpoints': checkpoints,
    'best_adapter_files': {
        path.name: {'bytes': path.stat().st_size, 'sha256': sha256(path)}
        for path in required
    },
    'selection_status': 'pending behavioral validation; loss-best is not automatically task-best',
    'final_test_status': 'untouched',
}
summary_path = RUN / 'training-completion-summary.json'
summary_path.write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))
print('Saved:', summary_path)


## After training

Send `training-completion-summary.json` and the last 100–200 lines of `full-console.log` back to Codex. The next step is to evaluate the base model and each retained epoch checkpoint on the same v5 validation cases, then select the behavioral winner. Do not run the sealed final test or the paid OpenAI comparison yet.
